# Working-memory study — annotated source notebook

A **self-contained, section-by-section version of the whole `Phillip/` pipeline** — the same code that lives in the `.py` files (`config`, `task`, `dynamics`, `reservoir_model`, `trained_rnn`, `analysis`, `plotting`, `run_experiment`), regrouped into small blocks with an explanation on top of each.

It exists for anyone who prefers to work **entirely inside Jupyter**. Unlike [`working_memory_project.ipynb`](working_memory_project.ipynb) — which *imports* the modules and narrates the experiment and its results — this notebook *defines every class and function inline*, so you never have to open a `.py` file.

### How to use it

1. Run **Setup** once (the imports).
2. Run sections **1–8** top to bottom. Almost every cell just *defines* things and runs instantly; a few short, clearly-labelled **Demo** cells show a piece working.
3. Once everything is defined you can build models, run analyses, and draw figures in your own cells — or jump to **Section 9** to run the entire study at once.

> **Tip:** `Kernel ▸ Restart & Run All` defines the complete pipeline in one go (the heavy end-to-end run in Section 9 is left commented out, so this stays fast).

> **Keeping it in sync:** this notebook is *generated from the `.py` files*, which remain the source of truth. If you change the code, edit the `.py` file and regenerate this notebook (or at least tell the group which copy is now authoritative) so the two don't drift apart.

## Setup

Every external library used anywhere in the pipeline, imported once. `Array = np.ndarray` is just a readability alias used in type hints throughout. `torch` is only needed by the Trained RNN in Section 5; everything else is plain NumPy.

In [1]:
import math
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

Array = np.ndarray  # readability alias for type hints

## 1. Shared experiment configuration

`ExperimentConfig` holds **every hyperparameter the two paradigms must agree on** — network size, task timing, the grid of delay lengths, and the standing noise floor. Both the Fixed Reservoir and the Trained RNN are built from the *same* `ExperimentConfig` instance, so any difference we measure later is attributable to **training**, not to a different task or a different network size. (Full rationale is in the docstring.)

In [2]:
@dataclass
class ExperimentConfig:
    """Hyperparameters shared by the reservoir and the trained RNN.

    Network
    -------
    N:
        Number of recurrent units. Chosen smaller than the base project's
        default (1000) so that the fully-trained RNN can be optimized with
        plain backpropagation-through-time on a laptop CPU in on the order of
        ten minutes for 1500 steps, rather than requiring a GPU, while still
        being far higher-dimensional than `n_classes` so a "collapse to a
        low-dimensional manifold" is a real, measurable compression rather
        than a foregone conclusion.
    g:
        Gain of the random recurrent matrix at initialization. Both
        paradigms start from this same value (see `init_shared_weights`);
        1.2 sits just past the classic edge-of-chaos point for a tanh
        network (Sompolinsky/Sussillo & Abbott), giving the un-trained
        reservoir rich, high-dimensional transient dynamics to work with.
    sp, gIn, spIn:
        Sparsity of the recurrent matrix, and gain/sparsity of the input
        projection. Same role as in `code/model_setup.py`.
    tau, dt:
        Membrane time constant and integration step, both in ms. `dt` is
        coarser than the base project's 0.1 ms (here 1.0 ms) purely to keep
        trial lengths (and therefore BPTT unroll length) computationally
        light; `tau=20` still spans 20 integration steps, which is enough to
        resolve the leaky dynamics.

    Task (cue-delay-recall)
    ------------------------
    n_classes:
        Number of distinct cue identities the network must remember.
    burn_in, cue_dur, recall_dur:
        Epoch durations in ms. `burn_in` is silence before the cue (lets the
        network settle from its initial condition); `cue_dur` is how long
        the one-hot cue is shown; `recall_dur` is how long the network is
        scored against the remembered identity once recall begins.
    delay_lengths:
        The grid of memory-delay durations (ms) trials are drawn from. This
        is the "varied delay lengths" axis in the abstract: both models are
        trained across this whole grid, and evaluated per-length to trace
        out a working-memory-capacity curve.
    go_pulse_dur:
        Duration (ms) of an explicit "go" input pulse at the start of the
        recall window (its own input channel, index `n_classes`). This
        signals *when* to report without requiring the network to also
        solve an elapsed-time-estimation problem, which is not the question
        under study here.

    Readout
    -------
    ridge_lambda:
        Ridge penalty for the reservoir's linear readout (see
        `reservoir_model.py`). Not used by the trained RNN, whose output
        weights are optimized jointly with everything else.

    Standing dynamical noise
    ------------------------
    intrinsic_noise_std:
        Gaussian noise added to the drive at *every* time step of *every*
        simulation (training, evaluation, and analysis alike) — a standing
        biological noise floor, distinct from the delay-only experimental
        perturbation. This matters for more than realism: the Fully Trained
        RNN is optimized against this same noisy simulator, so gradient
        descent only has a reason to prefer a noise-robust solution if noise
        was actually present during training. Without it, a fragile-but-
        accurate solution scores exactly as well as a robust one, and the
        robustness comparison this study makes would have no mechanism
        behind it. The delay-phase "additive recurrent noise" perturbation
        in `dynamics.Perturbation` adds *extra* noise on top of this
        baseline, specifically during the memory delay.
    """

    # --- network ---
    N: int = 300
    g: float = 1.2
    sp: float = 0.2
    gIn: float = 2.0
    spIn: float = 0.5
    tau: float = 20.0
    dt: float = 1.0

    # --- task / trial structure ---
    n_classes: int = 8
    burn_in: float = 5.0
    cue_dur: float = 10.0
    recall_dur: float = 15.0
    delay_lengths: tuple = (20.0, 80.0, 160.0, 300.0)
    go_pulse_dur: float = 5.0

    # --- reservoir readout ---
    ridge_lambda: float = 1.0

    # --- standing dynamical noise ---
    intrinsic_noise_std: float = 0.2

    @property
    def n_channels(self) -> int:
        """Input channels: one one-hot cue identity plus one go-pulse channel."""
        return self.n_classes + 1

    @property
    def primary_delay(self) -> float:
        """The longest scheduled delay: the main "deep dive" condition.

        Cross-temporal decoding, PCA trajectories, and perturbation sweeps
        are all run at this single delay length so that every trial in an
        analysis batch shares an identical timing/phase alignment (see
        `task.make_trial_batch`). The longest delay is also the most
        demanding maintenance condition, which is where representational
        differences between paradigms should be clearest.
        """
        return max(self.delay_lengths)

**Demo.** Instantiate the default config we'll use everywhere below.

In [3]:
config = ExperimentConfig()
config

ExperimentConfig(N=300, g=1.2, sp=0.2, gIn=2.0, spIn=0.5, tau=20.0, dt=1.0, n_classes=8, burn_in=5.0, cue_dur=10.0, recall_dur=15.0, delay_lengths=(20.0, 80.0, 160.0, 300.0), go_pulse_dur=5.0, ridge_lambda=1.0, intrinsic_noise_std=0.2)

## 2. The cue–delay–recall task

Each trial has four epochs laid out along the time axis:

```
burn-in | cue | delay (variable length) | recall
```

During **cue**, one of `n_classes` input channels is driven high (one-hot). During **delay** all inputs are silent — the network must carry the cue identity across the gap using only its own recurrent dynamics. **Recall** opens with a brief pulse on a dedicated *go* channel, after which the network is scored against the cued identity.

`to_steps` converts a duration in ms to an integer step count; `TrialBatch` is the container returned for a batch of trials.

In [4]:
def to_steps(duration_ms: float, dt: float) -> int:
    """Convert a duration in milliseconds to a (minimum 1) integer step count."""
    return max(1, int(round(duration_ms / dt)))


@dataclass
class TrialBatch:
    """A batch of cue-delay-recall trials sharing one time axis of length `T`.

    Shapes
    ------
    inputs:
        (n_trials, n_channels, T) — one-hot cue channels plus the go channel.
    targets:
        (n_trials, n_classes, T) — one-hot cued identity, held from recall
        onset through the end of the trial; all zero beforehand.
    cue_labels:
        (n_trials,) int — the cued class index for each trial.
    delay_length_ms:
        (n_trials,) float — the actual delay drawn for each trial (ms).
    cue_mask, delay_mask, recall_mask:
        (n_trials, T) bool — which time steps belong to which epoch, *per
        trial* (delay length can vary trial-to-trial even though `T` is
        shared, via padding; see module docstring).
    """

    inputs: Array
    targets: Array
    cue_labels: Array
    delay_length_ms: Array
    cue_mask: Array
    delay_mask: Array
    recall_mask: Array
    T: int
    dt: float

    @property
    def n_trials(self) -> int:
        return self.inputs.shape[0]

    @property
    def time_ms(self) -> Array:
        """Time axis in ms, with t=0 at burn-in start (matches `plotting.py`)."""
        return np.arange(self.T) * self.dt

`make_trial_batch` builds a batch. Passing a **single** delay length (`delay_lengths=(x,)`) gives every trial the same timing — required for the analyses in Section 6, which compare activity across trials at a shared time index. Passing the **full grid** (the default) draws an i.i.d. delay per trial, which is what training uses.

In [ ]:
def make_trial_batch(
    n_trials: int,
    config: ExperimentConfig,
    rng: np.random.Generator,
    delay_lengths: tuple | None = None,
) -> TrialBatch:
    """Generate a batch of cue-delay-recall trials.

    See the module docstring for when to pass a single-value `delay_lengths`
    (analysis) versus the full grid (training).
    """
    delay_lengths = config.delay_lengths if delay_lengths is None else delay_lengths # (20.0, 80.0, 160.0, 300.0)
    delay_choices = np.asarray(delay_lengths, dtype=float) # create np.array

    n_classes = config.n_classes # 8
    n_channels = config.n_channels #9
    go_channel = n_classes

    burn_in_steps = to_steps(config.burn_in, config.dt)
    cue_steps = to_steps(config.cue_dur, config.dt)
    recall_steps = to_steps(config.recall_dur, config.dt)
    go_steps = min(to_steps(config.go_pulse_dur, config.dt), recall_steps)
    max_delay_steps = to_steps(float(delay_choices.max()), config.dt)

    T = burn_in_steps + cue_steps + max_delay_steps + recall_steps

    inputs = np.zeros((n_trials, n_channels, T))
    targets = np.zeros((n_trials, n_classes, T))
    cue_mask = np.zeros((n_trials, T), dtype=bool)
    delay_mask = np.zeros((n_trials, T), dtype=bool)
    recall_mask = np.zeros((n_trials, T), dtype=bool)

    cue_labels = rng.integers(0, n_classes, size=n_trials) # Gives back one hot inputs 
    delay_length_ms = rng.choice(delay_choices, size=n_trials) # Gives back random delay choice

    cue_start = burn_in_steps
    cue_end = cue_start + cue_steps 

    for i in range(n_trials):
        delay_steps_i = to_steps(delay_length_ms[i], config.dt)
        delay_end = cue_end + delay_steps_i

        inputs[i, cue_labels[i], cue_start:cue_end] = 1.0
        inputs[i, go_channel, delay_end:delay_end + go_steps] = 1.0

        # Score from this trial's own recall onset through the shared end of
        # the batch. Trials with a shorter delay than the batch maximum
        # simply get a longer hold window; every trial is guaranteed at
        # least `recall_steps` of scored recall.
        targets[i, cue_labels[i], delay_end:T] = 1.0

        cue_mask[i, cue_start:cue_end] = True
        delay_mask[i, cue_end:delay_end] = True
        recall_mask[i, delay_end:T] = True

    return TrialBatch(
        inputs=inputs,
        targets=targets,
        cue_labels=cue_labels,
        delay_length_ms=delay_length_ms,
        cue_mask=cue_mask,
        delay_mask=delay_mask,
        recall_mask=recall_mask,
        T=T,
        dt=config.dt,
    )

In [28]:
config = ExperimentConfig()
rng = np.random.default_rng(0) # Gives back a GeneratorObject (new Version of np.random.seed)
print(rng)
some_test= rng.integers(0, 10, 5)
print(some_test)
batch = make_trial_batch(n_trials=2, config=config, rng=rng)
batch.inputs[0][0]

Generator(PCG64)
[8 6 5 2 3]


array([0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

**Demo.** Build two trials at the longest delay and inspect the shapes.

In [6]:
demo_rng = np.random.default_rng(0)
demo_batch = make_trial_batch(2, config, demo_rng, delay_lengths=(config.primary_delay,))
print("inputs  (trials, channels, time):", demo_batch.inputs.shape)
print("targets (trials, classes,  time):", demo_batch.targets.shape)
print("cued classes:", demo_batch.cue_labels)
print("trial length T =", demo_batch.T, "steps  (dt =", demo_batch.dt, "ms)")

inputs  (trials, channels, time): (2, 9, 330)
targets (trials, classes,  time): (2, 8, 330)
cued classes: [6 5]
trial length T = 330 steps  (dt = 1.0 ms)


## 3. Shared network dynamics

Both paradigms run through the **exact same** simulator once their weights exist — the only thing that differs between *reservoir* and *trained RNN* is which weights were shaped by gradient descent, never the simulation code. The recurrence is a leaky integrator:

$$r(t{+}\Delta t) = \tanh\!\big(u(t) + \alpha\,(r(t) - u(t))\big),\qquad u(t) = J\,r(t) + J_{in}\,x(t),\qquad \alpha = e^{-\Delta t/\tau}.$$

### 3a. Containers

`WeightBundle` is everything needed to simulate and read out one network instance. `Perturbation` describes a single delay-phase-only disruption (additive noise, weight degradation, or unit silencing).

In [7]:
@dataclass
class WeightBundle:
    """Everything needed to simulate and read out one network instance."""

    config: ExperimentConfig
    J: Array          # (N, N) recurrent weights
    Jin: Array         # (N, n_channels) input weights
    Wout: Array        # (n_classes, N) readout weights
    bout: Array        # (n_classes,) readout bias
    label: str          # e.g. "Fixed Reservoir" or "Trained RNN"; used in plots


@dataclass
class Perturbation:
    """A delay-phase-only perturbation applied by `simulate_trials`.

    Exactly one mechanism is active per instance:

    kind="noise":
        Gaussian noise with std `severity` is added to the recurrent+input
        drive at every delay time step ("additive recurrent noise").
    kind="degrade":
        `J_eff` (a corrupted copy of J) replaces J for the drive computation
        at every delay time step ("weight matrix degradation").
    kind="silence":
        Every unit flagged `True` in `mask` (shape (N,)) is held at zero
        firing rate at every delay time step ("unit silencing" — the same
        boolean mask is used whether it was chosen randomly or by a
        targeting criterion; see `analysis.py`).
    """

    kind: str
    severity: float = 0.0
    J_eff: Optional[Array] = None
    mask: Optional[Array] = None

### 3b. Weight initialization

`init_shared_weights` draws the random `(J, Jin)` used by **both** paradigms. The reservoir keeps them forever; the trained RNN starts from an identical copy and then reshapes them. This shared draw is what makes the comparison fair.

In [8]:
def create_weight_matrix(
    rows: int,
    columns: int,
    sparsity: float,
    gain: float,
    normalization_size: int,
    rng: np.random.Generator,
) -> Array:
    """Sparse random weight matrix with normalized variance.

    Identical scheme to `code/model_setup.py::create_weight_matrix`, just
    threaded through an explicit `Generator` instead of the numpy global
    random state, so weight initialization is reproducible independently of
    call order (important here since we draw many auxiliary random things —
    trial batches, perturbation instances — around the same model).
    """
    random_values = rng.normal(0.0, 1.0, size=(rows, columns))
    sparse_mask = rng.uniform(0.0, 1.0, size=(rows, columns)) <= sparsity
    return random_values * sparse_mask * gain / math.sqrt(normalization_size * sparsity)


def init_shared_weights(
    config: ExperimentConfig, rng: np.random.Generator
) -> tuple[Array, Array]:
    """Draw the random (J, Jin) initialization shared by BOTH paradigms.

    The reservoir keeps these forever and only trains a linear readout on
    top. The trained RNN starts from an identical copy of these same two
    matrices and then reshapes everything via gradient descent. Any
    representational difference measured later is therefore attributable to
    training, not to a different random architecture draw.
    """
    J = create_weight_matrix(
        config.N, config.N, config.sp, config.g, config.N, rng
    )
    Jin = create_weight_matrix(
        config.N, config.n_channels, config.spIn, config.gIn, config.n_channels, rng
    )
    return J, Jin

### 3c. The simulation loop

`simulate_trials` integrates the network forward over a whole batch, vectorized across trials (the only Python loop is over time, because the dynamics are recurrent). A perturbation, if given, is applied **only** while each trial is inside its own delay epoch — so any accuracy change is attributable specifically to disrupted *maintenance*.

In [9]:
def simulate_trials(
    bundle: WeightBundle,
    batch: TrialBatch,
    rng: np.random.Generator,
    perturbation: Optional[Perturbation] = None,
) -> Array:
    """Integrate the network forward over one trial batch.

    Returns firing rates with shape (n_trials, N, T). Vectorized over the
    trial dimension; the only unavoidable python loop is over time, since the
    dynamics are recurrent.
    """
    config = bundle.config
    n_trials = batch.n_trials
    N = config.N
    T = batch.T
    alpha = math.exp(-config.dt / config.tau)

    R = np.empty((n_trials, N, T))
    R[:, :, 0] = rng.uniform(0.0, 0.1, size=(n_trials, N))

    apply_noise = perturbation is not None and perturbation.kind == "noise"
    apply_degrade = perturbation is not None and perturbation.kind == "degrade"
    apply_silence = perturbation is not None and perturbation.kind == "silence"
    J_delay = perturbation.J_eff if apply_degrade else bundle.J

    for t in range(T - 1):
        r_t = R[:, :, t]
        x_t = batch.inputs[:, :, t]
        in_delay = batch.delay_mask[:, t]

        # Clean update (used outside the delay epoch, and as the baseline
        # for trials not currently perturbed within a mixed-condition call).
        # A standing intrinsic noise floor is always present (see
        # `ExperimentConfig.intrinsic_noise_std`) so that "additive recurrent
        # noise" as a delay-phase perturbation means genuinely *extra* noise
        # on top of a baseline both paradigms were already trained under.
        drive = r_t @ bundle.J.T + x_t @ bundle.Jin.T
        drive = drive + rng.normal(0.0, config.intrinsic_noise_std, size=drive.shape)
        r_next = np.tanh(drive + (r_t - drive) * alpha)

        if perturbation is not None and np.any(in_delay):
            drive_p = r_t @ J_delay.T + x_t @ bundle.Jin.T
            drive_p = drive_p + rng.normal(0.0, config.intrinsic_noise_std, size=drive_p.shape)
            if apply_noise:
                drive_p = drive_p + rng.normal(0.0, perturbation.severity, size=drive_p.shape)
            r_next_p = np.tanh(drive_p + (r_t - drive_p) * alpha)
            if apply_silence:
                r_next_p = r_next_p.copy()
                r_next_p[:, perturbation.mask] = 0.0
            r_next = np.where(in_delay[:, None], r_next_p, r_next)

        R[:, :, t + 1] = r_next

    return R

### 3d. Readout & scoring

Apply the linear readout at every time step, take the arg-max class, and score it against the cued identity within the recall window.

In [10]:
def decode_output(bundle: WeightBundle, R: Array) -> Array:
    """Apply the linear readout at every time step.

    R has shape (n_trials, N, T); returns class scores (n_trials, n_classes, T).
    """
    return np.einsum("cj,njt->nct", bundle.Wout, R) + bundle.bout[None, :, None]


def predicted_labels(scores: Array) -> Array:
    """Argmax over the class axis. scores: (n_trials, n_classes, T) -> (n_trials, T)."""
    return np.argmax(scores, axis=1)


def recall_accuracy(
    bundle: WeightBundle, R: Array, batch: TrialBatch
) -> float:
    """Fraction of (trial, time step) pairs correctly classified within `recall_mask`."""
    scores = decode_output(bundle, R)
    predicted = predicted_labels(scores)
    correct = predicted == batch.cue_labels[:, None]
    mask = batch.recall_mask
    if not np.any(mask):
        return float("nan")
    return float(correct[mask].mean())


def recall_accuracy_per_trial(
    bundle: WeightBundle, R: Array, batch: TrialBatch
) -> Array:
    """Per-trial recall accuracy (mean over that trial's own recall window)."""
    scores = decode_output(bundle, R)
    predicted = predicted_labels(scores)
    correct = (predicted == batch.cue_labels[:, None]) & batch.recall_mask
    counts = batch.recall_mask.sum(axis=1)
    return correct.sum(axis=1) / np.maximum(counts, 1)

## 4. Paradigm A — the Fixed Reservoir (Jaeger, 2001)

The recurrent connectivity is drawn once and **never trained**; only a linear readout is fit on top, in closed form via ridge regression. `fit_readout` pools every scored `(trial, time)` sample into one least-squares problem; `build_reservoir` packages the shared initialization as a reservoir and fits that readout.

In [11]:
def fit_readout(
    config: ExperimentConfig,
    firing_rates: Array,
    targets: Array,
    recall_mask: Array,
    regularization: float | None = None,
) -> tuple[Array, Array]:
    """Ridge-regression readout fit, pooling every (trial, time) sample
    inside `recall_mask` into one least-squares problem.

    ``firing_rates``: (n_trials, N, T). ``targets``: (n_trials, n_classes, T),
    one-hot. ``recall_mask``: (n_trials, T) bool. Returns ``(Wout, bout)``
    with shapes ``(n_classes, N)`` and ``(n_classes,)``.
    """
    regularization = config.ridge_lambda if regularization is None else regularization
    N = firing_rates.shape[1]

    # Boolean-index the (trial, time) leading dims to pool every scored
    # sample into one feature matrix, regardless of each trial's own
    # (possibly ragged) recall-window length.
    features = firing_rates.transpose(0, 2, 1)[recall_mask]        # (n_samples, N)
    target_samples = targets.transpose(0, 2, 1)[recall_mask]        # (n_samples, n_classes)

    # Constant bias feature, matching add_decoder_bias's role in the base project.
    features_b = np.hstack([features, np.ones((features.shape[0], 1))])

    gram = features_b.T @ features_b
    gram[np.diag_indices_from(gram)] += regularization
    weights = np.linalg.solve(gram, features_b.T @ target_samples)   # (N + 1, n_classes)

    Wout = weights[:N, :].T
    bout = weights[N, :]
    return Wout, bout


def build_reservoir(
    config: ExperimentConfig,
    J0: Array,
    Jin0: Array,
    rng: np.random.Generator,
    n_train_trials: int = 600,
) -> WeightBundle:
    """Package the shared random initialization as a fixed reservoir and fit
    its readout.

    ``J0``/``Jin0`` must be the same arrays passed to `trained_rnn.train_trained_rnn`
    for this comparison, so the two paradigms start from an identical
    architecture (see `dynamics.init_shared_weights`). Readout training uses
    trials with mixed delay lengths (the full grid in ``config.delay_lengths``),
    so the single fitted readout has to work across every delay duration —
    mirroring how the trained RNN is optimized in `trained_rnn.py`.
    """
    bundle = WeightBundle(
        config=config,
        J=J0,
        Jin=Jin0,
        Wout=np.zeros((config.n_classes, config.N)),
        bout=np.zeros(config.n_classes),
        label="Fixed Reservoir",
    )

    train_batch = make_trial_batch(n_train_trials, config, rng)
    firing_rates = simulate_trials(bundle, train_batch, rng)
    Wout, bout = fit_readout(config, firing_rates, train_batch.targets, train_batch.recall_mask)

    bundle.Wout = Wout
    bundle.bout = bout
    return bundle

## 5. Paradigm B — the Fully Trained RNN

Here **input, recurrent, and output weights are all optimized end-to-end** via gradient descent (backpropagation through time), in PyTorch. `LeakyRNN` is a deliberate, literal reimplementation of the same recurrence as `simulate_trials`, so the trained network's forward dynamics are identical to the reservoir's up to *which* weights gradients may touch. After training, the learned weights are exported back to a plain-NumPy `WeightBundle`, so every Section-6 analysis runs identically on both paradigms.

### 5a. Training hyperparameters

In [12]:
@dataclass
class TrainingConfig:
    """Hyperparameters of the optimization procedure itself.

    Kept separate from `ExperimentConfig`, which only holds the shared
    architecture/task substrate both paradigms must agree on. These values
    are specific to *how* the trained RNN is fit and have no reservoir
    counterpart.
    """

    n_steps: int = 1500
    batch_size: int = 64
    learning_rate: float = 2e-3
    weight_decay: float = 1e-5
    grad_clip_norm: float = 1.0
    n_eval_trials: int = 300
    eval_every: int = 100
    torch_seed: int = 0

### 5b. The trainable cell

Note the same standing noise floor (`intrinsic_noise_std`) is injected during training — that is what gives gradient descent an actual reason to prefer a noise-robust solution over a fragile-but-accurate one.

In [13]:
class LeakyRNN(nn.Module):
    """Trainable leaky-RNN cell plus linear readout, matching `dynamics.py`."""

    def __init__(self, config: ExperimentConfig, J0: Array, Jin0: Array):
        super().__init__()
        self.config = config
        self.alpha = math.exp(-config.dt / config.tau)
        self.J = nn.Parameter(torch.tensor(J0, dtype=torch.float32))
        self.Jin = nn.Parameter(torch.tensor(Jin0, dtype=torch.float32))
        self.Wout = nn.Parameter(torch.empty(config.n_classes, config.N))
        self.bout = nn.Parameter(torch.zeros(config.n_classes))
        nn.init.xavier_uniform_(self.Wout)

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """``inputs``: (n_trials, n_channels, T).

        Returns ``(rates, scores)`` with shapes ``(n_trials, N, T)`` and
        ``(n_trials, n_classes, T)``.
        """
        n_trials, _, T = inputs.shape
        r = torch.rand(n_trials, self.config.N) * 0.1
        rates = []
        for t in range(T):
            x_t = inputs[:, :, t]
            drive = r @ self.J.T + x_t @ self.Jin.T
            # Same standing noise floor as `dynamics.simulate_trials`
            # (`ExperimentConfig.intrinsic_noise_std`): training under this
            # noise is what gives gradient descent an actual reason to prefer
            # a noise-robust solution over a fragile-but-accurate one.
            if self.config.intrinsic_noise_std:
                drive = drive + torch.randn_like(drive) * self.config.intrinsic_noise_std
            r = torch.tanh(drive + (r - drive) * self.alpha)
            rates.append(r)
        rates = torch.stack(rates, dim=2)
        scores = torch.einsum("cn,bnt->bct", self.Wout, rates) + self.bout[None, :, None]
        return rates, scores

### 5c. The training loop

The loss is cross-entropy between the readout and the cued class, averaged over the recall window only — the network gets **no** direct supervision during cue or delay, so any useful delay dynamics must emerge purely from gradients flowing back through the delay.

In [14]:
def _masked_recall_accuracy(scores: torch.Tensor, cue_labels: Array, recall_mask: Array) -> float:
    predicted = scores.argmax(dim=1).numpy()
    correct = (predicted == cue_labels[:, None]) & recall_mask
    return float(correct.sum() / max(recall_mask.sum(), 1))


def train_trained_rnn(
    config: ExperimentConfig,
    J0: Array,
    Jin0: Array,
    rng: np.random.Generator,
    training: TrainingConfig | None = None,
) -> tuple[WeightBundle, dict]:
    """Train a `LeakyRNN` end-to-end and export it as a numpy `WeightBundle`.

    The loss is the cross-entropy between the readout scores and the cued
    class, averaged over every (trial, time step) pair inside that trial's
    own recall window (``batch.recall_mask``) — the network receives no
    direct supervision during the cue or delay epochs, so any useful
    delay-period dynamics must emerge purely from gradients that had to flow
    backward through the delay to make recall possible at all. Every
    training batch is drawn fresh with mixed delay lengths from
    ``config.delay_lengths``, so the network is pushed to maintain the cue
    across the *whole* delay grid, not just one duration.
    """
    training = training or TrainingConfig()
    torch.manual_seed(training.torch_seed)

    model = LeakyRNN(config, J0, Jin0)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=training.learning_rate, weight_decay=training.weight_decay
    )

    history = {"step": [], "loss": [], "eval_accuracy": []}

    for step in range(1, training.n_steps + 1):
        batch = make_trial_batch(training.batch_size, config, rng)
        inputs = torch.tensor(batch.inputs, dtype=torch.float32)
        cue_labels = torch.tensor(batch.cue_labels, dtype=torch.long)
        recall_mask = torch.tensor(batch.recall_mask)

        _, scores = model(inputs)
        target_bt = cue_labels[:, None].expand(-1, scores.shape[-1])
        loss_bt = F.cross_entropy(scores, target_bt, reduction="none")
        loss = (loss_bt * recall_mask).sum() / recall_mask.sum().clamp(min=1)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), training.grad_clip_norm)
        optimizer.step()

        if step % training.eval_every == 0 or step == training.n_steps:
            with torch.no_grad():
                eval_batch = make_trial_batch(training.n_eval_trials, config, rng)
                eval_inputs = torch.tensor(eval_batch.inputs, dtype=torch.float32)
                _, eval_scores = model(eval_inputs)
                acc = _masked_recall_accuracy(
                    eval_scores, eval_batch.cue_labels, eval_batch.recall_mask
                )
            history["step"].append(step)
            history["loss"].append(float(loss.item()))
            history["eval_accuracy"].append(acc)

    bundle = WeightBundle(
        config=config,
        J=model.J.detach().numpy().copy(),
        Jin=model.Jin.detach().numpy().copy(),
        Wout=model.Wout.detach().numpy().copy(),
        bout=model.bout.detach().numpy().copy(),
        label="Trained RNN",
    )
    return bundle, history

## 6. Analyses

Every function below takes a `WeightBundle` and only ever calls `simulate_trials` on it — it has no idea which paradigm produced the weights. That symmetry is what makes the comparison fair: identical task, identical analysis code, applied to both.

### 6.1 Cross-temporal ("temporal generalization") decoding

A ridge classifier is trained on population activity at each time $t_i$ and tested at every time $t_j$ (King & Dehaene, 2014). A temporally *stable* code generalizes off-diagonal (a wide bright square); a purely *dynamic* code only decodes near the diagonal. The **stability index** summarizes this as off-diagonal ÷ on-diagonal accuracy within the delay.

In [15]:
def _stratified_folds(cue_labels: Array, n_folds: int, rng: np.random.Generator) -> list:
    """Boolean test-masks (over trial index) for `n_folds` class-balanced folds."""
    fold_id = np.zeros(len(cue_labels), dtype=int)
    for c in np.unique(cue_labels):
        idx = np.where(cue_labels == c)[0]
        rng.shuffle(idx)
        fold_id[idx] = np.arange(len(idx)) % n_folds
    return [fold_id == k for k in range(n_folds)]


def _fit_ridge_classifier(
    features: Array, labels: Array, n_classes: int, ridge_lambda: float
) -> Array:
    """Closed-form ridge classifier. `features`: (n_samples, N). Returns W: (n_classes, N + 1)."""
    features_b = np.hstack([features, np.ones((features.shape[0], 1))])
    targets = np.zeros((features.shape[0], n_classes))
    targets[np.arange(features.shape[0]), labels] = 1.0
    gram = features_b.T @ features_b
    gram[np.diag_indices_from(gram)] += ridge_lambda
    weights = np.linalg.solve(gram, features_b.T @ targets)  # (N + 1, n_classes)
    return weights.T


def temporal_generalization_matrix(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    n_trials: int = 400,
    n_folds: int = 5,
    delay_length: Optional[float] = None,
    ridge_lambda: float = 5.0,
) -> dict:
    """Train-time x test-time cross-validated decoding accuracy (King & Dehaene, 2014).

    A ridge classifier is fit on population activity at every training time
    `t_i` (cross-validated across trials) and evaluated at every test time
    `t_j`. A code that is temporally *stable* (King & Dehaene's "stationary"
    regime) generalizes off the diagonal, producing a wide square of high
    accuracy; a purely *dynamic*, time-varying code only decodes well along
    the diagonal (t_i == t_j).
    """
    delay_length = config.primary_delay if delay_length is None else delay_length
    batch = make_trial_batch(n_trials, config, rng, delay_lengths=(delay_length,))
    R = simulate_trials(bundle, batch, rng)
    T = batch.T
    n_classes = config.n_classes

    folds = _stratified_folds(batch.cue_labels, n_folds, rng)
    acc_matrix = np.zeros((T, T))

    # Bias-augmented features at every time point, computed once: (T, n_trials, N + 1).
    features_b_all = np.concatenate(
        [R.transpose(2, 0, 1), np.ones((T, R.shape[0], 1))], axis=2
    )

    for test_mask in folds:
        train_mask = ~test_mask
        labels_train = batch.cue_labels[train_mask]
        labels_test = batch.cue_labels[test_mask]

        for t_train in range(T):
            W = _fit_ridge_classifier(
                R[train_mask, :, t_train], labels_train, n_classes, ridge_lambda
            )
            scores = np.einsum("cf,tnf->tnc", W, features_b_all[:, test_mask, :])
            predicted = np.argmax(scores, axis=2)  # (T, n_test)
            acc_matrix[t_train, :] += (predicted == labels_test[None, :]).mean(axis=1)

    acc_matrix /= len(folds)

    return {
        "accuracy": acc_matrix,
        "time_ms": batch.time_ms,
        "cue_mask": batch.cue_mask[0],
        "delay_mask": batch.delay_mask[0],
        "recall_mask": batch.recall_mask[0],
        "delay_length": delay_length,
        "label": bundle.label,
    }


def generalization_stability_index(result: dict) -> float:
    """Ratio of mean off-diagonal to mean on-diagonal decoding accuracy,
    restricted to the delay epoch.

    Near 1: a delay-period code that generalizes across time (temporally
    stable/stationary). Near 0: a code that only decodes well near its own
    training time (a genuinely time-varying/dynamic trajectory).
    """
    acc = result["accuracy"]
    delay_idx = np.where(result["delay_mask"])[0]
    if len(delay_idx) < 2:
        return float("nan")
    sub = acc[np.ix_(delay_idx, delay_idx)]
    diagonal = np.diag(sub)
    off_diagonal = sub[~np.eye(sub.shape[0], dtype=bool)]
    if diagonal.mean() <= 0:
        return float("nan")
    return float(off_diagonal.mean() / diagonal.mean())

### 6.2 PCA / dimensionality

PCA is fit on the full population-activity matrix pooled over every trial and time point. The **participation ratio** is a smooth, unit-free measure of effective dimensionality — roughly "how many PCs does this activity really need."

In [16]:
def _participation_ratio(pooled_centered: Array) -> float:
    """Effective dimensionality: PR = (sum(sigma^2))^2 / sum(sigma^4).

    Equals `n_components` if variance is spread equally over that many
    directions and 1.0 if a single direction explains everything; a smooth,
    unit-free way to summarize "how many PCs does this activity really need."
    """
    singular_values = np.linalg.svd(pooled_centered, full_matrices=False, compute_uv=False)
    power = singular_values ** 2
    return float((power.sum() ** 2) / (power ** 2).sum())


def pca_trajectories(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    n_trials: int = 240,
    delay_length: Optional[float] = None,
    n_components: int = 10,
) -> dict:
    """PCA on recurrent firing rates pooled across all simulated trials.

    Following the abstract's design, PCA is fit on the full population
    activity matrix pooled over every trial and time point (not on
    class-averaged activity, so genuine trial-to-trial variability shapes
    the components). For visualization, individual-trial trajectories are
    then projected into that same space and averaged within each cue class.
    """
    delay_length = config.primary_delay if delay_length is None else delay_length
    batch = make_trial_batch(n_trials, config, rng, delay_lengths=(delay_length,))
    R = simulate_trials(bundle, batch, rng)  # (n_trials, N, T)
    n_trials_actual, N, T = R.shape

    pooled = R.transpose(1, 0, 2).reshape(N, n_trials_actual * T)  # (N, n_trials * T)
    mean_ = pooled.mean(axis=1, keepdims=True)
    pooled_centered = pooled - mean_

    U, S, _ = np.linalg.svd(pooled_centered, full_matrices=False)
    components = U[:, :n_components]  # (N, n_components)

    projected = np.einsum("nk,bnt->bkt", components, R - mean_.reshape(1, N, 1))

    n_classes = config.n_classes
    class_trajectories = np.zeros((n_classes, n_components, T))
    for c in range(n_classes):
        class_trajectories[c] = projected[batch.cue_labels == c].mean(axis=0)

    variance_explained = (S ** 2) / np.sum(S ** 2)

    # Dimensionality restricted to the delay epoch specifically (the
    # maintenance period the abstract's hypothesis is about), computed from
    # its own centering rather than reusing the whole-trial one.
    delay_columns = np.tile(batch.delay_mask[0], n_trials_actual)
    pooled_delay = pooled[:, delay_columns]
    pooled_delay_centered = pooled_delay - pooled_delay.mean(axis=1, keepdims=True)

    return {
        "components": components,
        "projected_trials": projected,
        "class_trajectories": class_trajectories,
        "variance_explained": variance_explained,
        "participation_ratio": _participation_ratio(pooled_centered),
        "participation_ratio_delay": _participation_ratio(pooled_delay_centered),
        "cue_labels": batch.cue_labels,
        "time_ms": batch.time_ms,
        "cue_mask": batch.cue_mask[0],
        "delay_mask": batch.delay_mask[0],
        "recall_mask": batch.recall_mask[0],
        "delay_length": delay_length,
        "label": bundle.label,
    }

### 6.3 Lesion-target rankings

Two per-neuron importance scores used to choose *which* units to silence in the targeted-lesion experiment: a delay-period task-selectivity **F-statistic**, and each neuron's **decoder-weight magnitude**. Plus the helpers that turn a score (or randomness) into a boolean silencing mask.

In [17]:
def delay_selectivity_fstat(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    n_trials: int = 300,
    delay_length: Optional[float] = None,
) -> Array:
    """One-way ANOVA F-statistic per neuron on delay-averaged firing rate.

    High F means a neuron's mean activity during the memory delay strongly
    depends on which class was cued: a classic "memory neuron" signature,
    and a natural candidate criterion for targeted lesioning.
    """
    delay_length = config.primary_delay if delay_length is None else delay_length
    batch = make_trial_batch(n_trials, config, rng, delay_lengths=(delay_length,))
    R = simulate_trials(bundle, batch, rng)
    delay_mean = R[:, :, batch.delay_mask[0]].mean(axis=2)  # (n_trials, N)

    n_classes = config.n_classes
    grand_mean = delay_mean.mean(axis=0)
    ssb = np.zeros(config.N)
    ssw = np.zeros(config.N)
    for c in range(n_classes):
        group = delay_mean[batch.cue_labels == c]
        ssb += group.shape[0] * (group.mean(axis=0) - grand_mean) ** 2
        ssw += ((group - group.mean(axis=0)) ** 2).sum(axis=0)

    df_between = n_classes - 1
    df_within = max(n_trials - n_classes, 1)
    return (ssb / df_between) / (ssw / df_within + 1e-12)


def decoder_weight_magnitude(bundle: WeightBundle) -> Array:
    """L2 norm of each neuron's column in the readout: how heavily the
    linear decoder relies on that neuron."""
    return np.linalg.norm(bundle.Wout, axis=0)


def silencing_mask_from_ranking(scores: Array, fraction: float) -> Array:
    """Boolean mask silencing the top `fraction` of units by `scores` (highest first)."""
    N = len(scores)
    k = int(round(fraction * N))
    mask = np.zeros(N, dtype=bool)
    if k > 0:
        mask[np.argsort(scores)[-k:]] = True
    return mask


def random_silencing_mask(N: int, fraction: float, rng: np.random.Generator) -> Array:
    """Boolean mask silencing a uniformly random `fraction` of units."""
    k = int(round(fraction * N))
    mask = np.zeros(N, dtype=bool)
    if k > 0:
        mask[rng.choice(N, size=k, replace=False)] = True
    return mask

### 6.4 Perturbation sweeps

Each sweep re-evaluates recall accuracy while injecting a progressively stronger delay-phase disruption: additive **noise**, weight-matrix **degradation**, or unit **silencing** (random vs. targeted). `accuracy_vs_delay_length` is the unperturbed capacity baseline.

In [18]:
def _eval_accuracy(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    perturbation: Optional[Perturbation],
    n_trials: int,
    delay_length: float,
) -> float:
    batch = make_trial_batch(n_trials, config, rng, delay_lengths=(delay_length,))
    R = simulate_trials(bundle, batch, rng, perturbation=perturbation)
    return recall_accuracy(bundle, R, batch)


def noise_sweep(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    severities,
    n_trials: int = 300,
    n_repeats: int = 3,
    delay_length: Optional[float] = None,
) -> dict:
    """Recall accuracy vs. additive Gaussian noise injected into the
    delay-phase drive (severity = noise standard deviation)."""
    delay_length = config.primary_delay if delay_length is None else delay_length
    means, stds = [], []
    for severity in severities:
        accs = [
            _eval_accuracy(
                config, bundle, rng, Perturbation(kind="noise", severity=severity),
                n_trials, delay_length,
            )
            for _ in range(n_repeats)
        ]
        means.append(np.mean(accs))
        stds.append(np.std(accs))
    return {
        "severities": np.asarray(severities, dtype=float),
        "accuracy_mean": np.asarray(means),
        "accuracy_std": np.asarray(stds),
        "label": bundle.label,
    }


def degradation_sweep(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    severities,
    n_trials: int = 300,
    n_repeats: int = 3,
    delay_length: Optional[float] = None,
) -> dict:
    """Recall accuracy vs. structural noise added to J during the delay
    (severity = added-noise std, in units of the existing weight std)."""
    delay_length = config.primary_delay if delay_length is None else delay_length
    weight_scale = float(np.std(bundle.J))
    means, stds = [], []
    for severity in severities:
        accs = []
        for _ in range(n_repeats):
            J_eff = bundle.J + rng.normal(0.0, severity * weight_scale, size=bundle.J.shape)
            pert = Perturbation(kind="degrade", severity=severity, J_eff=J_eff)
            accs.append(_eval_accuracy(config, bundle, rng, pert, n_trials, delay_length))
        means.append(np.mean(accs))
        stds.append(np.std(accs))
    return {
        "severities": np.asarray(severities, dtype=float),
        "accuracy_mean": np.asarray(means),
        "accuracy_std": np.asarray(stds),
        "label": bundle.label,
    }


def silencing_sweep(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    fractions,
    targeting: str,
    ranking_scores: Optional[Array] = None,
    n_trials: int = 300,
    n_repeats: int = 3,
    delay_length: Optional[float] = None,
) -> dict:
    """Recall accuracy vs. fraction of units silenced during the delay.

    ``targeting="random"`` draws a fresh random mask each repeat.
    ``targeting="targeted"`` always silences the top-scoring units from
    ``ranking_scores`` (e.g. `delay_selectivity_fstat` or
    `decoder_weight_magnitude`); repeats still resample test trials and
    initial-condition jitter, so they remain informative even though the
    silenced set itself is fixed.
    """
    if targeting not in ("random", "targeted"):
        raise ValueError("targeting must be 'random' or 'targeted'")
    if targeting == "targeted" and ranking_scores is None:
        raise ValueError("targeted silencing requires ranking_scores")

    delay_length = config.primary_delay if delay_length is None else delay_length
    means, stds = [], []
    for fraction in fractions:
        accs = []
        for _ in range(n_repeats):
            mask = (
                random_silencing_mask(config.N, fraction, rng)
                if targeting == "random"
                else silencing_mask_from_ranking(ranking_scores, fraction)
            )
            pert = Perturbation(kind="silence", severity=fraction, mask=mask)
            accs.append(_eval_accuracy(config, bundle, rng, pert, n_trials, delay_length))
        means.append(np.mean(accs))
        stds.append(np.std(accs))
    return {
        "fractions": np.asarray(fractions, dtype=float),
        "accuracy_mean": np.asarray(means),
        "accuracy_std": np.asarray(stds),
        "targeting": targeting,
        "label": bundle.label,
    }


def accuracy_vs_delay_length(
    config: ExperimentConfig,
    bundle: WeightBundle,
    rng: np.random.Generator,
    n_trials: int = 300,
) -> dict:
    """Baseline (unperturbed) recall accuracy at each scheduled delay length."""
    means = []
    for delay in config.delay_lengths:
        batch = make_trial_batch(n_trials, config, rng, delay_lengths=(delay,))
        R = simulate_trials(bundle, batch, rng)
        means.append(recall_accuracy(bundle, R, batch))
    return {
        "delay_lengths": np.asarray(config.delay_lengths, dtype=float),
        "accuracy": np.asarray(means),
        "label": bundle.label,
    }

## 7. Plotting

Figure-generating functions. Every function returns a Matplotlib `Figure` rather than calling `plt.show()`. Colour is fixed for the whole study: the **Fixed Reservoir is always blue, the Trained RNN always green**.

### 7a. Palette & shared helpers

In [19]:
MODEL_COLORS = {
    "Fixed Reservoir": "#2a78d6",   # categorical slot 1 (blue)
    "Trained RNN": "#008300",        # categorical slot 2 (green)
}


CLASS_COLORS = [
    "#2a78d6",  # slot 1 blue
    "#008300",  # slot 2 green
    "#e87ba4",  # slot 3 magenta
    "#eda100",  # slot 4 yellow
    "#1baf7a",  # slot 5 aqua
    "#eb6834",  # slot 6 orange
    "#4a3aa7",  # slot 7 violet
    "#e34948",  # slot 8 red
]


SEQUENTIAL_BLUE_STEPS = [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b",
]


INK_PRIMARY = "#0b0b0b"


INK_SECONDARY = "#52514e"


INK_MUTED = "#898781"


GRIDLINE = "#e1e0d9"


SURFACE = "#fcfcfb"


def get_pyplot():
    """Import matplotlib only when plotting is actually needed."""
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "matplotlib is required for plotting. Install it with `pip install matplotlib`."
        ) from exc
    return plt


def _sequential_blue_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("sequential_blue", SEQUENTIAL_BLUE_STEPS)


def _style_axis(ax):
    """Recessive grid/axes: light gridlines, muted ticks, no top/right spine."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(INK_MUTED)
    ax.spines["bottom"].set_color(INK_MUTED)
    ax.tick_params(colors=INK_SECONDARY)
    ax.yaxis.label.set_color(INK_PRIMARY)
    ax.xaxis.label.set_color(INK_PRIMARY)
    ax.title.set_color(INK_PRIMARY)
    ax.grid(True, color=GRIDLINE, linewidth=0.8, alpha=0.9)
    ax.set_axisbelow(True)


def _mark_epochs(ax, batch_like, vertical=True):
    """Dashed boundary lines + small labels at cue/delay/recall transitions.

    ``batch_like`` needs ``time_ms``, ``cue_mask``, ``delay_mask``, ``recall_mask``
    (1-D, one fixed-delay trial's worth — as returned by the analysis functions).
    """
    time_ms = batch_like["time_ms"]
    cue_end = time_ms[batch_like["cue_mask"]][-1] if np.any(batch_like["cue_mask"]) else None
    delay_end = time_ms[batch_like["delay_mask"]][-1] if np.any(batch_like["delay_mask"]) else None
    boundaries = [t for t in (cue_end, delay_end) if t is not None]
    line_fn = ax.axvline if vertical else ax.axhline
    for t in boundaries:
        line_fn(t, color=INK_MUTED, linestyle="--", linewidth=1.0, alpha=0.8)

### 7b. Task, weights & capacity figures

The trial schematic, the recurrent-weight / eigenvalue comparison, the RNN training curve, and the accuracy-vs-delay capacity curve.

In [20]:
def plot_trial_schematic(batch, trial_idx: int, config) -> "Figure":
    """Inputs and recall target for one example trial, with epoch boundaries."""
    plt = get_pyplot()
    fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

    time_ms = batch.time_ms
    inputs = batch.inputs[trial_idx]
    targets = batch.targets[trial_idx]

    ax[0].imshow(
        inputs, aspect="auto", cmap=_sequential_blue_cmap(),
        extent=[time_ms[0], time_ms[-1], inputs.shape[0] - 0.5, -0.5],
    )
    ax[0].set_yticks(range(inputs.shape[0]))
    ax[0].set_yticklabels([f"cue {i}" for i in range(config.n_classes)] + ["go"])
    ax[0].set_title(f"Example trial (delay = {batch.delay_length_ms[trial_idx]:.0f} ms)")

    ax[1].imshow(
        targets, aspect="auto", cmap=_sequential_blue_cmap(),
        extent=[time_ms[0], time_ms[-1], targets.shape[0] - 0.5, -0.5],
    )
    ax[1].set_yticks(range(targets.shape[0]))
    ax[1].set_yticklabels([f"class {i}" for i in range(config.n_classes)])
    ax[1].set_xlabel("time (ms)")
    ax[1].set_ylabel("recall target")
    ax[0].set_ylabel("input channel")

    single = {
        "time_ms": time_ms,
        "cue_mask": batch.cue_mask[trial_idx],
        "delay_mask": batch.delay_mask[trial_idx],
        "recall_mask": batch.recall_mask[trial_idx],
    }
    for a in ax:
        _mark_epochs(a, single)
        a.set_facecolor(SURFACE)

    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def plot_weight_spectrum_comparison(bundles: list) -> "Figure":
    """Sample of J plus its eigenvalue spectrum, side by side per model.

    Shows directly what gradient descent did to the recurrent matrix
    relative to the shared random initialization the reservoir kept as-is.
    """
    plt = get_pyplot()
    fig, ax = plt.subplots(2, len(bundles), figsize=(6 * len(bundles), 9))
    if len(bundles) == 1:
        ax = ax[:, None]

    show_count = 50
    for col, bundle in enumerate(bundles):
        color = MODEL_COLORS.get(bundle.label, INK_PRIMARY)
        sample = ax[0, col].imshow(
            bundle.J[:show_count, :show_count], cmap=_sequential_blue_cmap()
        )
        ax[0, col].set_title(f"{bundle.label}: sample of J", color=color)
        ax[0, col].set_xlabel("presynaptic neuron")
        ax[0, col].set_ylabel("postsynaptic neuron")
        plt.colorbar(sample, ax=ax[0, col], fraction=0.046, pad=0.04)

        eigenvalues = np.linalg.eigvals(bundle.J)
        circle = np.linspace(0, 2 * np.pi, 200)
        ax[1, col].plot(np.cos(circle), np.sin(circle), color=INK_MUTED, linewidth=1.0)
        ax[1, col].scatter(
            np.real(eigenvalues), np.imag(eigenvalues), s=10, color=color, alpha=0.7
        )
        ax[1, col].set_title(f"{bundle.label}: eigenvalue spectrum", color=color)
        ax[1, col].set_xlabel("real component")
        ax[1, col].set_ylabel("imaginary component")
        ax[1, col].set_aspect("equal")
        _style_axis(ax[1, col])

    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def plot_training_curve(history: dict) -> "Figure":
    """Loss and held-out recall accuracy across training, as two separate
    axes (never dual-axis — the two quantities have unrelated scales)."""
    plt = get_pyplot()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    color = MODEL_COLORS["Trained RNN"]
    ax[0].plot(history["step"], history["loss"], color=color, linewidth=2)
    ax[0].set_xlabel("training step")
    ax[0].set_ylabel("recall-window cross-entropy loss")
    ax[0].set_title("Training loss")

    ax[1].plot(history["step"], history["eval_accuracy"], color=color, linewidth=2)
    ax[1].axhline(1.0, color=INK_MUTED, linestyle=":", linewidth=1)
    ax[1].set_ylim(0, 1.05)
    ax[1].set_xlabel("training step")
    ax[1].set_ylabel("held-out recall accuracy")
    ax[1].set_title("Training progress")

    for a in ax:
        _style_axis(a)
        a.set_facecolor(SURFACE)
    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def plot_accuracy_vs_delay(results: list) -> "Figure":
    """Recall accuracy vs. delay length, one line per model."""
    plt = get_pyplot()
    fig, ax = plt.subplots(figsize=(6.5, 5))

    for result in results:
        color = MODEL_COLORS.get(result["label"], INK_PRIMARY)
        ax.plot(
            result["delay_lengths"], result["accuracy"], marker="o", markersize=6,
            color=color, linewidth=2, label=result["label"],
        )

    ax.set_ylim(0, 1.05)
    ax.set_xlabel("delay length (ms)")
    ax.set_ylabel("recall accuracy")
    ax.set_title("Working-memory accuracy across delay lengths")
    ax.legend(frameon=False, labelcolor=INK_PRIMARY)
    _style_axis(ax)
    ax.set_facecolor(SURFACE)
    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig

### 7c. Representation & robustness figures

The cross-temporal decoding heatmaps, the PCA trajectories, the dimensionality bar chart, and the three perturbation-sweep plots.

In [21]:
def plot_temporal_generalization(results: list) -> "Figure":
    """Train-time x test-time decoding accuracy heatmaps, one panel per model."""
    plt = get_pyplot()
    fig, ax = plt.subplots(1, len(results), figsize=(6.5 * len(results), 5.5))
    if len(results) == 1:
        ax = [ax]

    for col, result in enumerate(results):
        time_ms = result["time_ms"]
        extent = [time_ms[0], time_ms[-1], time_ms[-1], time_ms[0]]
        im = ax[col].imshow(
            result["accuracy"], extent=extent, cmap=_sequential_blue_cmap(),
            vmin=0, vmax=1, aspect="equal",
        )
        color = MODEL_COLORS.get(result["label"], INK_PRIMARY)
        stability = result.get("stability_index")
        title = f"{result['label']}"
        if stability is not None:
            title += f"  (stability = {stability:.2f})"
        ax[col].set_title(title, color=color)
        ax[col].set_xlabel("test time (ms)")
        ax[col].set_ylabel("train time (ms)")
        single = {
            "time_ms": time_ms,
            "cue_mask": result["cue_mask"],
            "delay_mask": result["delay_mask"],
            "recall_mask": result["recall_mask"],
        }
        _mark_epochs(ax[col], single, vertical=True)
        _mark_epochs(ax[col], single, vertical=False)
        plt.colorbar(im, ax=ax[col], fraction=0.046, pad=0.04, label="decoding accuracy")

    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def plot_pca_trajectories(results: list) -> "Figure":
    """Per-class trajectories in PC1-PC2 space (top row) and PC1 over time
    (bottom row), one column per model."""
    plt = get_pyplot()
    fig, ax = plt.subplots(2, len(results), figsize=(6.5 * len(results), 10))
    if len(results) == 1:
        ax = ax[:, None]

    n_classes = results[0]["class_trajectories"].shape[0]
    for col, result in enumerate(results):
        traj = result["class_trajectories"]  # (n_classes, n_components, T)
        time_ms = result["time_ms"]
        color = MODEL_COLORS.get(result["label"], INK_PRIMARY)

        top = ax[0, col]
        for c in range(n_classes):
            class_color = CLASS_COLORS[c % len(CLASS_COLORS)]
            top.plot(traj[c, 0], traj[c, 1], color=class_color, linewidth=2, alpha=0.9)
            top.scatter(traj[c, 0, 0], traj[c, 1, 0], color=class_color, marker="o", s=40, zorder=3)
            top.scatter(traj[c, 0, -1], traj[c, 1, -1], color=class_color, marker="s", s=40, zorder=3)
            top.annotate(str(c), (traj[c, 0, -1], traj[c, 1, -1]), color=INK_PRIMARY,
                         fontsize=9, fontweight="bold", ha="center", va="center")
        top.set_title(f"{result['label']}: state-space trajectories", color=color)
        top.set_xlabel("PC1")
        top.set_ylabel("PC2")

        bottom = ax[1, col]
        for c in range(n_classes):
            class_color = CLASS_COLORS[c % len(CLASS_COLORS)]
            bottom.plot(time_ms, traj[c, 0], color=class_color, linewidth=2, alpha=0.9, label=f"class {c}")
        single = {
            "time_ms": time_ms, "cue_mask": result["cue_mask"],
            "delay_mask": result["delay_mask"], "recall_mask": result["recall_mask"],
        }
        _mark_epochs(bottom, single)
        bottom.set_xlabel("time (ms)")
        bottom.set_ylabel("PC1")
        bottom.set_title(f"{result['label']}: PC1 over time", color=color)
        if col == len(results) - 1:
            bottom.legend(frameon=False, labelcolor=INK_PRIMARY, fontsize=8, loc="upper right")

        for a in (top, bottom):
            _style_axis(a)
            a.set_facecolor(SURFACE)

    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def plot_dimensionality_summary(results: list) -> "Figure":
    """Grouped bar chart: participation ratio, whole-trial vs. delay-only, per model."""
    plt = get_pyplot()
    fig, ax = plt.subplots(figsize=(6.5, 5))

    groups = ["Whole trial", "Delay only"]
    x = np.arange(len(groups))
    width = 0.35 if len(results) == 2 else 0.6 / max(len(results), 1)

    for i, result in enumerate(results):
        color = MODEL_COLORS.get(result["label"], INK_PRIMARY)
        values = [result["participation_ratio"], result["participation_ratio_delay"]]
        offset = (i - (len(results) - 1) / 2) * width
        ax.bar(x + offset, values, width=width, color=color, label=result["label"])

    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    ax.set_ylabel("participation ratio (effective # of PCs)")
    ax.set_title("Dimensionality of population activity")
    ax.legend(frameon=False, labelcolor=INK_PRIMARY)
    _style_axis(ax)
    ax.set_facecolor(SURFACE)
    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def _sweep_plot(results: list, x_key: str, xlabel: str, title: str) -> "Figure":
    plt = get_pyplot()
    fig, ax = plt.subplots(figsize=(6.5, 5))

    for result in results:
        color = MODEL_COLORS.get(result["label"], INK_PRIMARY)
        x = result[x_key]
        mean = result["accuracy_mean"]
        std = result["accuracy_std"]
        ax.plot(x, mean, marker="o", markersize=5, color=color, linewidth=2, label=result["label"])
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.15, linewidth=0)

    ax.set_ylim(0, 1.05)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("recall accuracy")
    ax.set_title(title)
    ax.legend(frameon=False, labelcolor=INK_PRIMARY)
    _style_axis(ax)
    ax.set_facecolor(SURFACE)
    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig


def plot_noise_sweep(results: list) -> "Figure":
    return _sweep_plot(
        results, "severities", "recurrent noise std (added during delay)",
        "Robustness to additive recurrent noise",
    )


def plot_degradation_sweep(results: list) -> "Figure":
    return _sweep_plot(
        results, "severities", "weight-matrix degradation (x weight std, during delay)",
        "Robustness to weight-matrix degradation",
    )


def plot_silencing_comparison(sweep_by_targeting: dict) -> "Figure":
    """One panel per targeting strategy; each panel compares both models.

    ``sweep_by_targeting``: {targeting_name: [reservoir_result, rnn_result]}.
    Column order and titles come from the dict's insertion order.
    """
    plt = get_pyplot()
    n_panels = len(sweep_by_targeting)
    fig, ax = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5), sharey=True)
    if n_panels == 1:
        ax = [ax]

    for col, (targeting_name, results) in enumerate(sweep_by_targeting.items()):
        for result in results:
            color = MODEL_COLORS.get(result["label"], INK_PRIMARY)
            x = result["fractions"]
            mean = result["accuracy_mean"]
            std = result["accuracy_std"]
            ax[col].plot(x, mean, marker="o", markersize=5, color=color, linewidth=2, label=result["label"])
            ax[col].fill_between(x, mean - std, mean + std, color=color, alpha=0.15, linewidth=0)
        ax[col].set_ylim(0, 1.05)
        ax[col].set_xlabel("fraction of units silenced")
        ax[col].set_title(targeting_name)
        if col == 0:
            ax[col].set_ylabel("recall accuracy")
            ax[col].legend(frameon=False, labelcolor=INK_PRIMARY)
        _style_axis(ax[col])
        ax[col].set_facecolor(SURFACE)

    fig.suptitle("Robustness to unit silencing during the delay", color=INK_PRIMARY)
    fig.patch.set_facecolor(SURFACE)
    fig.tight_layout()
    return fig

## 8. Orchestration

The glue that runs the whole study: build both paradigms, run every analysis, save the figures and a numeric summary. (These are the functions from `run_experiment.py`, with the `analysis.` / `plotting.` module prefixes removed so they call the definitions above directly — making this notebook fully standalone.)

### 8a. Output locations & sweep grids

In [22]:
# A notebook has no __file__, so we assume the working directory is Phillip/.
PROJECT_ROOT = Path.cwd()
PLOTS_DIR = PROJECT_ROOT / "plots"
RESULTS_DIR = PROJECT_ROOT / "results"

GLOBAL_SEED = 0

# Perturbation strengths swept in the robustness analyses.
NOISE_SEVERITIES = [0.0, 0.1, 0.2, 0.4, 0.8, 1.2, 2.0]
DEGRADATION_SEVERITIES = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
SILENCING_FRACTIONS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]

### 8b. Build, analyze, save

In [23]:
def save_fig(fig, filename: str, plots_dir: Path = PLOTS_DIR) -> Path:
    plots_dir.mkdir(exist_ok=True, parents=True)
    path = plots_dir / filename
    fig.savefig(path, format="svg", bbox_inches="tight")
    return path


def build_and_train(
    config: ExperimentConfig,
    rng: np.random.Generator,
    training: TrainingConfig | None = None,
    n_reservoir_train_trials: int = 600,
):
    """Draw the shared initialization, then build both paradigms from it."""
    J0, Jin0 = init_shared_weights(config, rng)
    reservoir = build_reservoir(config, J0.copy(), Jin0.copy(), rng, n_reservoir_train_trials)
    rnn_bundle, history = train_trained_rnn(config, J0.copy(), Jin0.copy(), rng, training)
    return reservoir, rnn_bundle, history


def run_full_analysis(config: ExperimentConfig, bundles: list, rng: np.random.Generator) -> dict:
    """Run every analysis (decoding, PCA, perturbation sweeps) on each bundle."""
    results = {"delay_curve": [], "generalization": [], "pca": [], "noise": [],
               "degradation": [], "silencing": {}, "selectivity": {}, "decoder_weight": {}}

    silencing_by_targeting = {
        "Random silencing": [],
        "Targeted: task-selectivity": [],
        "Targeted: decoder weight": [],
    }

    for bundle in bundles:
        results["delay_curve"].append(accuracy_vs_delay_length(config, bundle, rng))

        gen = temporal_generalization_matrix(config, bundle, rng)
        gen["stability_index"] = generalization_stability_index(gen)
        results["generalization"].append(gen)

        results["pca"].append(pca_trajectories(config, bundle, rng))

        fstat = delay_selectivity_fstat(config, bundle, rng)
        weight_mag = decoder_weight_magnitude(bundle)
        results["selectivity"][bundle.label] = fstat
        results["decoder_weight"][bundle.label] = weight_mag

        results["noise"].append(noise_sweep(config, bundle, rng, NOISE_SEVERITIES))
        results["degradation"].append(degradation_sweep(config, bundle, rng, DEGRADATION_SEVERITIES))

        silencing_by_targeting["Random silencing"].append(
            silencing_sweep(config, bundle, rng, SILENCING_FRACTIONS, targeting="random")
        )
        silencing_by_targeting["Targeted: task-selectivity"].append(
            silencing_sweep(
                config, bundle, rng, SILENCING_FRACTIONS, targeting="targeted", ranking_scores=fstat
            )
        )
        silencing_by_targeting["Targeted: decoder weight"].append(
            silencing_sweep(
                config, bundle, rng, SILENCING_FRACTIONS, targeting="targeted", ranking_scores=weight_mag
            )
        )

    results["silencing"] = silencing_by_targeting
    return results


def save_all_figures(config: ExperimentConfig, bundles: list, history: dict, results: dict, rng: np.random.Generator):
    example_batch = make_trial_batch(4, config, rng, delay_lengths=(config.primary_delay,))
    save_fig(plot_trial_schematic(example_batch, 0, config), "trial_schematic.svg")

    save_fig(plot_weight_spectrum_comparison(bundles), "weight_spectrum_comparison.svg")
    save_fig(plot_training_curve(history), "rnn_training_curve.svg")
    save_fig(plot_accuracy_vs_delay(results["delay_curve"]), "accuracy_vs_delay.svg")
    save_fig(plot_temporal_generalization(results["generalization"]), "temporal_generalization.svg")
    save_fig(plot_pca_trajectories(results["pca"]), "pca_trajectories.svg")
    save_fig(plot_dimensionality_summary(results["pca"]), "dimensionality_summary.svg")
    save_fig(plot_noise_sweep(results["noise"]), "perturbation_noise.svg")
    save_fig(plot_degradation_sweep(results["degradation"]), "perturbation_degradation.svg")
    save_fig(plot_silencing_comparison(results["silencing"]), "perturbation_silencing.svg")


def summarize(bundles: list, results: dict) -> dict:
    """Collapse the full result set into plain-Python scalars for summary.json."""
    summary = {}
    for i, bundle in enumerate(bundles):
        label = bundle.label
        summary[label] = {
            "accuracy_by_delay": dict(zip(
                results["delay_curve"][i]["delay_lengths"].tolist(),
                results["delay_curve"][i]["accuracy"].tolist(),
            )),
            "generalization_stability_index": results["generalization"][i]["stability_index"],
            "participation_ratio_whole_trial": results["pca"][i]["participation_ratio"],
            "participation_ratio_delay": results["pca"][i]["participation_ratio_delay"],
            "noise_severities": results["noise"][i]["severities"].tolist(),
            "noise_accuracy": results["noise"][i]["accuracy_mean"].tolist(),
            "degradation_severities": results["degradation"][i]["severities"].tolist(),
            "degradation_accuracy": results["degradation"][i]["accuracy_mean"].tolist(),
        }
        for targeting_name, sweep_list in results["silencing"].items():
            summary[label][f"silencing[{targeting_name}]_fractions"] = sweep_list[i]["fractions"].tolist()
            summary[label][f"silencing[{targeting_name}]_accuracy"] = sweep_list[i]["accuracy_mean"].tolist()
    return summary


def main() -> None:
    rng = np.random.default_rng(GLOBAL_SEED)
    config = ExperimentConfig()

    print("Building shared initialization and training both paradigms...")
    reservoir, rnn_bundle, history = build_and_train(config, rng)
    bundles = [reservoir, rnn_bundle]

    print("Reservoir readout fit. Trained RNN final eval accuracy:", history["eval_accuracy"][-1])

    print("Running full analysis suite (decoding CV + PCA + perturbation sweeps for both models; this can take ~15-20 minutes on CPU)...")
    results = run_full_analysis(config, bundles, rng)

    print("Saving figures to", PLOTS_DIR)
    save_all_figures(config, bundles, history, results, rng)

    print("Saving numeric summary to", RESULTS_DIR)
    RESULTS_DIR.mkdir(exist_ok=True, parents=True)
    with open(RESULTS_DIR / "summary.json", "w") as f:
        json.dump(summarize(bundles, results), f, indent=2)

    print("Done.")

## 9. Running it

### Quick start (fast) — build the reservoir and check its capacity

This trains **no** RNN (the reservoir only fits a linear readout), so it finishes in a few seconds and reproduces the reservoir's headline result: near-perfect recall at short delays, decaying at the longest one.

In [24]:
rng = np.random.default_rng(GLOBAL_SEED)
config = ExperimentConfig()

J0, Jin0 = init_shared_weights(config, rng)
reservoir = build_reservoir(config, J0.copy(), Jin0.copy(), rng)

acc = accuracy_vs_delay_length(config, reservoir, rng, n_trials=100)
print("Fixed Reservoir — recall accuracy by delay length "
      f"(chance = {1 / config.n_classes:.3f})")
for d, a in zip(acc["delay_lengths"], acc["accuracy"]):
    print(f"  {d:6.0f} ms : {a:.3f}")

Fixed Reservoir — recall accuracy by delay length (chance = 0.125)
      20 ms : 0.999
      80 ms : 0.983
     160 ms : 0.980
     300 ms : 0.704


### The full study

Uncomment `main()` below to build **both** paradigms, run every analysis and perturbation sweep, and write all figures to `plots/` plus `results/summary.json` — identical to running `python run_experiment.py` from the shell. This trains the RNN from scratch and takes roughly **15–25 minutes** on a laptop CPU, so it is left commented out.

In [25]:
# main()